In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path='/content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path='./'

In [ ]:
#import
from torch import nn,optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from matplotlib import pyplot as plt
import os

DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'current device: {DEVICE}')

current device: cuda


In [ ]:
#Hyperparameter
BATCH_SIZE=32
EPOCH=20
LR=1e-3
new_model_train = True

criterion=nn.CrossEntropyLoss()
model_type="CNN"
dataset="CIFAR_10"

print(f"{model_type}_{dataset}")

save_model_path = f"./{model_type}_{dataset}.pt" #모델 저장

CNN_CIFAR_10


In [ ]:
root=os.path.join(path, 'data')
# transform=transforms.ToTensor()
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# train_DS=datasets.CIFAR10(root=root, train=True, download= True, transform=transform)
# test_DS=datasets.CIFAR10(root=root, train=False, download= True, transform=transform)
# full_train_DS=datasets.CIFAR10(root='./data', train=True, download= True, transform=transform)
# test_DS=datasets.CIFAR10(root='./data', train=False, download= True, transform=transform)
full_train_DS = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_DS = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)


train_size = 45000
val_size = 5000
train_DS, val_DS = random_split(full_train_DS, [train_size, val_size])

train_DL=torch.utils.data.DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = torch.utils.data.DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=False) # 검증용 추가
test_DL=torch.utils.data.DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded: Train({len(train_DS)}), Val({len(val_DS)}), Test({len(test_DS)})")

Data loaded: Train(45000), Val(5000), Test(10000)


In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # self.conv1=nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        # self.maxPool=nn.MaxPool2d(kernel_size=2, stride=2)
        # self.conv2=nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv_layers=nn.Sequential( nn.Conv2d(3,64,3,padding=1),
                                        nn.BatchNorm2d(64),
                                        nn.ReLU(),
                                        nn.Conv2d(64,64,3,padding=1),
                                        nn.BatchNorm2d(64),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2), # 32x32 -> 16x16

                                        nn.Conv2d(64,128,3,padding=1),
                                        nn.BatchNorm2d(128),
                                        nn.ReLU(),
                                        nn.Conv2d(128,128,3,padding=1),
                                        nn.BatchNorm2d(128),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2), # 16x16 -> 8x8

                                        nn.Conv2d(128,256,3,padding=1),
                                        nn.BatchNorm2d(256),
                                        nn.ReLU(),
                                        nn.Conv2d(256,256,3,padding=1),
                                        nn.BatchNorm2d(256),
                                        nn.ReLU(),
                                        nn.Conv2d(256,256,3,padding=1),
                                        nn.BatchNorm2d(256),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2)) # 8x8 -> 4x4
        self.flat=nn.Flatten(start_dim=1)
        self.fc_layers=nn.Sequential( nn.Linear(256*4*4,1024),
                                      nn.ReLU(),
                                      nn.Dropout(0.5),
                                      nn.Linear(1024,256),
                                      nn.ReLU(),
                                      nn.Dropout(0.5),
                                      nn.Linear(256,10))

    def forward(self, x):
        x=self.conv_layers(x)
        x=self.flat(x)
        x=self.fc_layers(x)
        return x

In [ ]:
CNN_model=MyModel().to(DEVICE)

In [ ]:
checkpoint_dir = os.path.join(path, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)
def Train(model, train_DL, val_DL, EPOCH, criterion, optimizer, DEVICE):
    save_every = 5
    save_best = True

    best_val_acc = -1.0

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    n_train = len(train_DL.dataset)
    n_val = len(val_DL.dataset)

    for ep in range(EPOCH):
        model.train()
        train_loss = 0
        train_acc = 0

        for x_batch, y_batch in train_DL:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            # Forward
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            # Backward & Update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x_batch.size(0)
            train_acc += (y_pred.argmax(dim=1) == y_batch).sum().item()

        #Validation
        model.eval()
        val_loss = 0
        val_acc = 0

        with torch.no_grad():
            for x_batch, y_batch in val_DL:
                x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)

                y_pred = model(x_batch)
                loss = criterion(y_pred, y_batch)

                val_loss += loss.item() * x_batch.size(0)
                val_acc += (y_pred.argmax(dim=1) == y_batch).sum().item()

        ep_train_loss = train_loss / n_train
        ep_train_acc = train_acc / n_train
        ep_val_loss = val_loss / n_val
        ep_val_acc = val_acc / n_val

        history['train_loss'].append(ep_train_loss)
        history['train_acc'].append(ep_train_acc)
        history['val_loss'].append(ep_val_loss)
        history['val_acc'].append(ep_val_acc)


        # 체크포인트 저장(주기적)
        if save_every > 0 and (ep + 1) % save_every == 0:
            ckpt_path = os.path.join(checkpoint_dir, f'ckpt_ep{ep+1}.pt')
            torch.save({
                'epoch': ep + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'val_acc': ep_val_acc
            }, ckpt_path)
            print(f"ckpt_ep{ep+1}.pt saved")

        # 베스트 모델 저장
        if save_best and ep_val_acc > best_val_acc:
            best_val_acc = ep_val_acc
            best_path = os.path.join(checkpoint_dir, 'best_model.pt')
            torch.save({
                'epoch': ep + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': ep_val_acc
            }, best_path)
            # print(f"cbest_model.pt saved")

        check_ep= EPOCH//10 if EPOCH>10 else 1
        if (ep+1) % check_ep ==0:
            print(f"Epoch [{ep+1}/{EPOCH}] "
                f"Train Loss: {ep_train_loss:.4f} | Acc: {ep_train_acc*100:.2f}% | "
                f"Val Loss: {ep_val_loss:.4f} | Acc: {ep_val_acc*100:.2f}%")

    return history

In [ ]:
optimizer=optim.Adam(CNN_model.parameters(), lr=LR, weight_decay=1e-4)
history = Train(CNN_model, train_DL, val_DL, EPOCH, criterion, optimizer, DEVICE)
torch.save(CNN_model.state_dict(), os.path.join(checkpoint_dir,'CNN_CIFAR10_final_weights.pth'))
torch.save(history, os.path.join(checkpoint_dir, 'history.pt'))

NameError: name 'optim' is not defined

In [ ]:
# 모델 불러오기 (아래처럼 모델 인스턴스 생성 후)
model = MyModel().to(DEVICE)
model.load_state_dict(torch.load(os.path.join(checkpoint_dir,'CNN_CIFAR10_final_weights.pth'), map_location=DEVICE))
model.eval()

# 체크포인트 복원 예시 (옵티마이저까지 복원)
ckpt = torch.load(os.path.join(checkpoint_dir, 'best_model.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt.get('epoch', 0)

In [ ]:
def plot_history(history):
    plt.figure(figsize=(12, 4))

    # Loss 그래프
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss Trend')
    plt.xlabel('Epoch')
    plt.legend()

    # Accuracy 그래프
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Accuracy Trend')
    plt.xlabel('Epoch')
    plt.legend()

    plt.show()

plot_history(history)

NameError: name 'history' is not defined

In [ ]:
def Test(model, test_DL):
    model.eval() # test mode로 전환
    with torch.no_grad(): #model.eval과 같이 항상 해야 함
        rcorrect = 0
        for x_batch, y_batch in test_DL:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            # inference
            y_hat = model(x_batch)
            # corrects accumulation
            pred = y_hat.argmax(dim=1)
            corrects_b = torch.sum(pred == y_batch).item() # torch.eq(pred, y_batch).sum().item()
            rcorrect += corrects_b
        accuracy_e = rcorrect/len(test_DL.dataset)*100
    print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")
    return round(accuracy_e,1)

In [ ]:
Test(CNN_model, test_DL)

Test accuracy: 8403/10000 (84.0 %)


84.0

In [ ]:
cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    def __init__(self, vgg_name):
        super(VGG, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)

In [ ]:
net = VGG('VGG11')
net = net.to(DEVICE)
EPOCH=30

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=1e-2,
                      momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCH)

history= Train(net, train_DL, val_DL, EPOCH, criterion, optimizer, DEVICE)


Epoch [3/30] Train Loss: 0.8984 | Acc: 68.77% | Val Loss: 0.8760 | Acc: 69.84%
Epoch [6/30] Train Loss: 0.6345 | Acc: 78.04% | Val Loss: 0.6692 | Acc: 77.66%
Epoch [9/30] Train Loss: 0.5155 | Acc: 82.37% | Val Loss: 0.5445 | Acc: 81.80%
Epoch [12/30] Train Loss: 0.4486 | Acc: 84.74% | Val Loss: 0.5716 | Acc: 81.42%
Epoch [15/30] Train Loss: 0.4086 | Acc: 86.00% | Val Loss: 0.5323 | Acc: 82.94%
Epoch [18/30] Train Loss: 0.3746 | Acc: 87.17% | Val Loss: 0.5769 | Acc: 80.58%
Epoch [21/30] Train Loss: 0.3523 | Acc: 87.99% | Val Loss: 0.6052 | Acc: 80.70%
Epoch [24/30] Train Loss: 0.3292 | Acc: 88.70% | Val Loss: 0.5056 | Acc: 82.84%
Epoch [27/30] Train Loss: 0.3143 | Acc: 89.07% | Val Loss: 0.4916 | Acc: 83.62%
Epoch [30/30] Train Loss: 0.3047 | Acc: 89.55% | Val Loss: 0.5109 | Acc: 83.96%


In [ ]:
EPOCH=10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCH)
history2= Train(net, train_DL, val_DL, EPOCH, criterion, optimizer, DEVICE)

Epoch [1/10] Train Loss: 0.3022 | Acc: 89.67% | Val Loss: 0.4836 | Acc: 84.30%
Epoch [2/10] Train Loss: 0.2987 | Acc: 89.69% | Val Loss: 0.5546 | Acc: 82.62%
Epoch [3/10] Train Loss: 0.2962 | Acc: 89.84% | Val Loss: 0.4644 | Acc: 84.60%
Epoch [4/10] Train Loss: 0.2974 | Acc: 89.75% | Val Loss: 0.4761 | Acc: 84.58%
Epoch [5/10] Train Loss: 0.2951 | Acc: 89.92% | Val Loss: 0.4960 | Acc: 83.64%
Epoch [6/10] Train Loss: 0.2853 | Acc: 90.11% | Val Loss: 0.5230 | Acc: 83.30%
Epoch [7/10] Train Loss: 0.2869 | Acc: 90.28% | Val Loss: 0.4571 | Acc: 85.10%
Epoch [8/10] Train Loss: 0.2858 | Acc: 90.12% | Val Loss: 0.5149 | Acc: 83.34%
Epoch [9/10] Train Loss: 0.2855 | Acc: 90.28% | Val Loss: 0.4783 | Acc: 84.52%
Epoch [10/10] Train Loss: 0.2808 | Acc: 90.25% | Val Loss: 0.5147 | Acc: 83.58%
